# Evaluating a Deep Agent

## Setting the stage

A single run can look convincing and still hide a brittle system. The agent labelled three
tickets correctly &mdash; because the policy is sound, or because that particular phrasing
happened to work? And when you swap the model underneath it, does anything actually
improve, or do you just get a different set of plausible-looking answers?

Those are measurement questions, and one demo cannot answer them. This notebook turns a
**support-ticket triage** agent into a repeatable offline LangSmith experiment, then runs
that experiment **twice** against the same dataset and the same three evaluators &mdash; once
on Anthropic's `claude-haiku-4-5-20251001` and once on OpenAI's `gpt-5.4-mini` &mdash; to see
whether swapping the model underneath a working agent changes anything the evaluators can
actually detect.

<img src="images/triage-agent-architecture.svg"
     alt="One inbound support ticket enters a supervisor. The supervisor delegates to a triage-specialist subagent, which consults a lookup_policy tool and writes a strictly shaped /triage.json artifact. The supervisor reads that artifact back and reports a one-line brief for the on-call engineer. Below a dashed line labelled Offline Evaluation, the JSON artifact is graded by the code evaluators schema_valid and triage_accuracy, while the prose brief is graded by the LLM judge summary_quality."
     style="width:100%; height:auto; border:1px solid #d0d7de; border-radius:6px;" />

The split across the bottom of that diagram is the thing to hold on to. The specialist
leaves behind `/triage.json`, which is strictly shaped and can be checked by ordinary code.
The supervisor leaves behind prose, which cannot. Two agents, two artifacts &mdash; and so
two different kinds of evaluator:

- **`schema_valid`** and **`triage_accuracy`** are ordinary code. They read the JSON
  artifact the specialist wrote and check it against the ticket and the reference answer.
  No model is involved, so the score is exact and costs nothing.
- **`summary_quality`** is an LLM judge. It grades the supervisor's human-facing brief for
  something code cannot see: whether it stayed faithful to what the specialist actually
  produced.

The policy and the tickets are fixed rather than live. That keeps the lesson reproducible:
when a score moves, the agent moved &mdash; the world did not.

In [28]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
import json
from typing import Annotated, TypedDict

from dotenv import load_dotenv
load_dotenv(override=True)

from IPython.display import Code, Markdown, display
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langsmith import Client
from deepagents import create_deep_agent
from util.triage_dataset import LABELS, PRIORITIES, TRIAGE_EXAMPLES

AGENT_MODELS = ["anthropic:claude-haiku-4-5-20251001", "openai:gpt-5.4-mini"]  # the axis we vary
JUDGE_MODEL = "anthropic:claude-sonnet-5"                                      # held fixed on purpose

client = Client()

# The scenario: a ticket and a fixed policy

Support triage is a good subject for evaluation because correctness is mostly not a matter
of taste. A policy says what the labels mean and how priority follows from them, and a
ticket either matches that policy or it does not.

The policy below is the whole world our agent gets. It is exposed as a tool rather than
baked into the prompt, so the specialist has to go and read it &mdash; the shape a real
triage agent has against a policy that changes without a redeploy.

The six tickets are deliberately not evenly easy. Four of them are written so that the
surface phrasing points at the wrong answer: a question about whether something is safe that
is not a security incident, a payment failure delivered as an angry bug report, a real
credential exposure reported in perfectly calm language, and a defect phrased as a how-to
question. `T-102` and `T-402` are the sharpest pair &mdash; both are bugs, and their priorities
differ only because one customer mentions in passing that they have found a way to keep
working.

Every one of them is still decidable &mdash; that is what the tie-breakers in the policy are
for. Nothing here is a matter of taste, which is the property that lets `triage_accuracy`
be arithmetic rather than opinion.

In [30]:
TRIAGE_POLICY = """\
# Support triage policy

## Labels
- billing   -- payments, invoices, refunds, card declines, subscription charges.
- how-to    -- the product behaves as designed and the user needs guidance.
- bug       -- the product does not behave as documented.
- security  -- credentials or data have been exposed, or an account can still be reached
               by someone who should no longer have access.

## Tie-breakers
- A payment that fails is billing, however the user chooses to describe it.
- Asking whether something is safe or supported is how-to. It becomes security only once
  credentials or data have actually been exposed.
- A report of wrong behaviour is a bug even when it is phrased as a question.

## Priority
- P1 -- any security ticket, or a bug with no workaround the customer can use.
- P2 -- any billing ticket, or a bug where the customer describes a way to keep working.
- P3 -- any how-to ticket.
"""

@tool
def lookup_policy() -> str:
    """Return the current support triage policy: label definitions and priority rules."""
    return TRIAGE_POLICY

# A specialist with one job, and a supervisor that reports

Triage is a clean subagent boundary. The supervisor never classifies anything; it hands the
ticket to a specialist and then reports what came back.

Those two roles leave behind two different artifacts, and keeping them apart is what makes
the system gradeable at all:

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:10px; text-align:left;" width="24%">Layer</th>
<th style="border:1px solid #999; padding:10px; text-align:left;" width="38%">Produces</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Graded by</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><code style="color:#2b8a3e;">triage-specialist</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><code style="color:#2b8a3e;">/triage.json</code> &mdash; one strictly shaped answer per ticket</td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Ordinary code. The structure and the labels are exact, so a model would only add
cost and noise.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>supervisor</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">A short brief for the on-call engineer</td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">An LLM judge. No assertion can ask whether prose still means what the file
meant.</td>
</tr>
</tbody>
</table>

That second question is a genuine deep-agent failure mode. A supervisor summarizing a
subagent's work is exactly where a ticket quietly acquires a priority nobody assigned it.

Note that the supervisor is *not* asked to echo the file. The old temptation is to make the
handoff a verbatim copy so a string comparison can check it, but that turns the supervisor
into a no-op and grades nothing worth grading.

In [31]:
TRIAGE_PATH = "/triage.json"

TRIAGE_SCHEMA = """{
  "id": "T-000",
  "label": "one of: billing, how-to, bug, security",
  "priority": "one of: P1, P2, P3",
  "reason": "one sentence naming the policy rule you applied"
}"""

triage_specialist = {
    "name": "triage-specialist",
    "description": (
        "Classify one inbound support ticket against the triage policy, "
        f"write the result to {TRIAGE_PATH}, and return only that path."
    ),
    "system_prompt": (
        "You are a support triage specialist. Call lookup_policy before classifying anything, "
        "and apply it exactly as written rather than by intuition. Classify the ticket you "
        f"are given, reusing its id exactly. Write {TRIAGE_PATH} as raw JSON matching this "
        "schema, with no code fence and no prose around it, then return only the path.\n\n"
        + TRIAGE_SCHEMA
    ),
    "tools": [lookup_policy],
}

SUPERVISOR_PROMPT = (
    "You are the support triage lead. Delegate every inbound ticket to the triage-specialist "
    f"subagent. When it returns, read {TRIAGE_PATH} and write one short line for the on-call "
    "engineer: the ticket id, the label and priority exactly as the file gives them, and the "
    "next action to take. Do not restate the customer's complaint, and do not mention the "
    "file or its path."
)

def build_supervisor(model: str):
    """One deep agent per model under test: same prompts, same tools, different model."""
    return create_deep_agent(
        model=model,
        system_prompt=SUPERVISOR_PROMPT,
        subagents=[triage_specialist],
    )

# The target: what LangSmith runs

A bit of vocabulary: A **dataset** holds **examples**, each one pairing an `inputs` object with a
`reference outputs` object. An **experiment** is a single pass of a single configuration over
every example, turning each into one **run**. **Evaluators** score those runs, and each score
comes back as feedback attached to the run it graded.

<img src="images/offline-experiment-anatomy.svg"
     alt="A dataset box named deepagents-support-triage-evals-v3 holds six example rows, each pairing one ticket as inputs with a label and priority as reference outputs. An arrow labelled one call per example leads down into a dashed container labelled Experiment, described as one pass of one configuration over every example. Inside it a chain runs left to right: target(inputs), then a Run holding outputs and a trace, then the Evaluators schema_valid, triage_accuracy and summary_quality, then Feedback, one score per evaluator attached to its run. A footnote marks target(inputs) and reads: the target is typically the agent itself, wrapped just enough to accept a dataset example and hand the evaluators what they need."
     style="width:100%; height:auto; border:1px solid #d0d7de; border-radius:6px;" />

The **target** is the blue box in the middle &mdash; the system under test, an ordinary
function LangSmith calls once for every example. Everything to its left is data you wrote;
everything to its right is measurement.

Ours is a *factory* rather than a plain function, because the configuration is the thing we
want to vary. `make_target(model)` builds a supervisor on that model and hands back the
function that runs it. One call per configuration, one dataset &mdash; and that is the whole
mechanism behind the two experiments at the end.

The target does three things and nothing else:

1. Read the ticket off the dataset input.
2. Invoke the supervisor exactly as the application would.
3. Adapt deep-agent state into a stable `{brief, triage, artifact}` shape.

Step 3 is where the JSON parsing lives, so that evaluators receive data rather than a
transcript to mine. The target grades nothing and never sees the reference labels.

Run it once before writing a single evaluator, and look at what actually comes back.
Extraction logic written against an *imagined* output shape is the classic way to spend a
whole experiment discovering a typo.

In [32]:
def format_ticket(ticket: dict) -> str:
    return (
        "Triage this support ticket.\n\n"
        f"{ticket['id']} - {ticket['subject']}: {ticket['body']}"
    )


def parse_triage(artifact: str | None) -> dict | None:
    """Parse the specialist's artifact. Validating its contents is schema_valid's job."""
    if not isinstance(artifact, str) or not artifact.strip():
        return None
    body = artifact.strip()
    if body.startswith("```"):                    # models occasionally fence their JSON
        body = body.split("```")[1].removeprefix("json").strip()
    try:
        parsed = json.loads(body)
    except ValueError:
        return None
    return parsed if isinstance(parsed, dict) else None


def make_target(model: str):
    """Build the function LangSmith calls once per dataset example."""
    supervisor = build_supervisor(model)

    def run_triage(inputs: dict) -> dict:
        result = supervisor.invoke(
            {"messages": [{"role": "user", "content": format_ticket(inputs)}]}
        )
        files = result.get("files") or {}
        artifact = (files.get(TRIAGE_PATH) or {}).get("content")
        return {
            "brief": result["messages"][-1].text,   # what the on-call engineer reads
            "triage": parse_triage(artifact),       # what the code evaluators grade
            "artifact": artifact,                   # kept raw so failures are debuggable
        }

    return run_triage

In [33]:
DEMO_TICKET = TRIAGE_EXAMPLES[0]["inputs"]   # the dataset's first example

demo = make_target(AGENT_MODELS[0])(DEMO_TICKET)

display(Markdown("### `brief` &mdash; what the on-call engineer reads"))
display(Markdown(demo["brief"]))
display(Markdown(f"### `triage` &mdash; parsed from `{TRIAGE_PATH}`"))
display(Code(json.dumps(demo["triage"], indent=2), language="json"))

### `brief` &mdash; what the on-call engineer reads

**T-101 | billing | P2** — Verify duplicate charge and process refund if confirmed.

### `triage` &mdash; parsed from `/triage.json`

{
  "id": "T-101",
  "label": "billing",
  "priority": "P2",
  "reason": "Duplicate charge and refund request fall under billing issues per policy."
}

# Turn expectations into a dataset

Each example is one ticket:

- `inputs` is the ticket itself &mdash; its id, subject, and body.
- `outputs` is the reference classification &mdash; the label and priority the policy requires
  for it.

That one-in, one-out shape is what LangSmith examples usually look like, and it is worth
holding to. Every row in the experiment view is then one ticket, so a failing score points at
a specific ticket instead of at a batch you then have to read a comment to decode.

The reference is a lookup table, not an expected response. Nothing in it says how the
supervisor should word its brief, because many wordings are fine and only the
classification has one right answer.

The rows themselves live in `data/support_tickets.jsonl`, one JSON object per line in exactly
the `{inputs, outputs}` shape LangSmith wants. `util/triage_dataset.py` is a short loader over
that file, and it checks as it reads: every reference answer has to follow from the policy's
priority rules, so the dataset cannot quietly drift out of agreement with the policy the agent
is handed.

Each row also carries `metadata` marking whether it is one of the four tickets whose surface
phrasing points at the wrong answer. That travels into LangSmith with the example, so an
experiment can be grouped by `trap` in the UI &mdash; which is how you tell "the model is weak"
apart from "the model is only weak on the tickets designed to fool it".

The dataset name is versioned, so re-running the notebook reuses the existing dataset
rather than deleting remote data.

In [34]:
DATASET_NAME = "deepagents-support-triage-evals-v3"

if client.has_dataset(dataset_name=DATASET_NAME):
    dataset = client.read_dataset(dataset_name=DATASET_NAME)
else:
    dataset = client.create_dataset(
        DATASET_NAME,
        description="Policy-following and supervisor-faithfulness evals for a support triage agent.",
    )
    client.create_examples(dataset_id=dataset.id, examples=TRIAGE_EXAMPLES)

display(Markdown(
    f"**{len(TRIAGE_EXAMPLES)} tickets, one per example** &mdash; [{DATASET_NAME}]({dataset.url})"
))
display(Markdown(
    "One whole example &mdash; the reference is a lookup table, not an expected response:"
))
display(Code(json.dumps(TRIAGE_EXAMPLES[0], indent=2), language="json"))

**6 tickets, one per example** &mdash; [deepagents-support-triage-evals-v3](https://smith.langchain.com/o/60995f7e-6a4c-4170-99c5-ad8ee020b23f/datasets/b1fb508e-5f28-4220-ac81-c78ff4cf53c1)

One whole example &mdash; the reference is a lookup table, not an expected response:

{
  "inputs": {
    "id": "T-101",
    "subject": "Charged twice for October",
    "body": "My card was charged $49 twice on Oct 3. Please refund one of them."
  },
  "outputs": {
    "label": "billing",
    "priority": "P2"
  },
  "metadata": {
    "trap": false,
    "note": "a straightforward duplicate charge"
  }
}

# Three evaluators, three questions

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:10px; text-align:left;" width="22%">Evaluator</th>
<th style="border:1px solid #999; padding:10px; text-align:left;" width="14%">Kind</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Question</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><code style="color:#2b8a3e;">schema_valid</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Code</td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Did the specialist write a well-formed answer for this exact ticket, using only
known labels and priorities?</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><code style="color:#2b8a3e;">triage_accuracy</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Code</td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Do the label and priority match the answer the policy requires?</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><code style="color:#2b8a3e;">summary_quality</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">LLM judge</td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Does the supervisor's brief still say what the file says, in a form the on-call
engineer can act on?</td>
</tr>
</tbody>
</table>

One evaluator, one metric. Keeping them apart is what makes a failing experiment
actionable: `schema_valid` failing means the output contract broke, `triage_accuracy`
failing means the policy was misapplied, and `summary_quality` failing means the handoff
between the two agents lost something on the way out.

Note also that `triage_accuracy` is the only score that is not binary: it grades two fields,
so a ticket can come back half right. Averaged over the six examples it separates a model
that fumbles one edge case from one that has misread the policy outright.

In [35]:
def schema_valid(inputs: dict, outputs: dict) -> dict:
    """Did the specialist produce a well-formed answer for this exact ticket?"""
    triage = outputs.get("triage")
    if not isinstance(triage, dict):
        return {
            "key": "schema_valid",
            "score": 0,
            "comment": f"No parseable JSON object at {TRIAGE_PATH}.",
        }

    problems = []
    if triage.get("id") != inputs["id"]:
        problems.append(f"id={triage.get('id')!r}, expected {inputs['id']!r}")
    if triage.get("label") not in LABELS:
        problems.append(f"label={triage.get('label')!r}")
    if triage.get("priority") not in PRIORITIES:
        problems.append(f"priority={triage.get('priority')!r}")

    return {
        "key": "schema_valid",
        "score": int(not problems),
        "comment": "; ".join(problems) or "id, label and priority all present and valid.",
    }


GRADED_FIELDS = ("label", "priority")


def triage_accuracy(outputs: dict, reference_outputs: dict) -> dict:
    """Do the label and priority match the answer the policy requires?"""
    triage = outputs.get("triage") or {}
    misses = [
        f"{field}: wanted {reference_outputs[field]}, got {triage.get(field)!r}"
        for field in GRADED_FIELDS
        if triage.get(field) != reference_outputs[field]
    ]
    return {
        "key": "triage_accuracy",
        "score": (len(GRADED_FIELDS) - len(misses)) / len(GRADED_FIELDS),
        "comment": "; ".join(misses) or "label and priority both match.",
    }

# The judge: faithfulness, not taste

`summary_quality` asks three yes/no questions about the supervisor's brief, and all three
have to pass. None of them is about whether the brief reads nicely.

The judge is handed the ticket, the specialist's `/triage.json`, and the brief. It is
deliberately **not** handed the reference labels &mdash; grading those is
`triage_accuracy`'s job. Splitting them this way means a run where the specialist mislabels
a ticket and the supervisor reports that mislabel faithfully scores `triage_accuracy` low
and `summary_quality` 1, which is exactly the diagnosis you want: the policy reasoning is
broken, the handoff is not.

<div style="background:#fff8e1; border:2px solid #f0ad4e; border-radius:6px; padding:14px 18px; font-size:16px;">
<b>&#9888;&#65039; Pin the judge model.</b><br/><br/>
The two experiments below vary the model the <i>agent</i> runs on. The judge stays on
<code style="color:#2b8a3e;">JUDGE_MODEL</code> for both. A grader that moves with the system
under test is not measuring the same thing twice, and the comparison it produces means
nothing.
</div>

In [36]:
class SummaryGrade(TypedDict):
    faithful: Annotated[
        bool,
        "The label and priority stated in the brief match the triage file exactly, and the "
        "brief describes the same ticket the file does.",
    ]
    actionable: Annotated[
        bool,
        "The brief names a next action for the on-call engineer rather than restating the "
        "customer's complaint.",
    ]
    reasoning: Annotated[str, "One or two sentences naming each failed criterion."]


SUMMARY_CHECKS = ("faithful", "actionable")

summary_judge_prompt = """You grade a support triage lead's one-line brief against the triage
file it was written from. You are grading fidelity and usefulness, not prose style.

Set faithful=true only when the ticket id, label and priority stated in the brief match the triage
file exactly, and the brief does not describe a ticket the file says nothing about.

Set actionable=true only when the brief tells the on-call engineer what to do next, rather than
only restating what the customer said.

Do not judge whether the label itself is correct. A separate evaluator does that, and a brief that
faithfully reports a wrong label is still faithful."""

# No temperature=0 here: claude-sonnet-5 rejects the parameter outright.
judge = init_chat_model(JUDGE_MODEL).with_structured_output(SummaryGrade)


def summary_quality(inputs: dict, outputs: dict) -> dict:
    """Does the supervisor's brief still say what the specialist's file says?"""
    brief = outputs.get("brief")
    if not isinstance(brief, str) or not brief.strip():
        return {
            "key": "summary_quality",
            "score": 0,
            "comment": "The supervisor returned no brief.",
        }

    grade = judge.invoke([
        SystemMessage(content=summary_judge_prompt),
        HumanMessage(content=(
            f"Ticket:\n{format_ticket(inputs)}\n\n"
            f"Triage file ({TRIAGE_PATH}):\n{json.dumps(outputs.get('triage'), indent=2)}\n\n"
            f"Supervisor brief:\n{brief}"
        )),
    ])

    checks = ", ".join(
        f"{name}={'pass' if grade[name] else 'fail'}" for name in SUMMARY_CHECKS
    )
    return {
        "key": "summary_quality",
        "score": int(all(grade[name] for name in SUMMARY_CHECKS)),
        "comment": f"{checks}. {grade['reasoning']}",
    }

# Two experiments, one dataset

Everything so far describes a single configuration. An **experiment** is what makes one
configuration comparable to another.

The two runs below differ in exactly one thing: the model the supervisor and its specialist
run on. One is Anthropic's small, fast tier and the other is OpenAI's, which makes this a
comparison of *providers* rather than of capability &mdash; the question is not which model is
larger, but whether swapping the engine underneath a working agent moves anything you would
care about.

Note what does not change. The dataset, the three evaluators, and the judge are all held
still, and `JUDGE_MODEL` stays on one Anthropic model for both runs. A grader that drifted
along with the system under test would not be measuring the same thing twice &mdash; and that
matters more, not less, once the two systems come from different vendors.

On this dataset the comparison tends to come back **flat**: the task sits well within reach of
both models. A result like that is easy to misread as a broken harness, so it is worth being
blunt about it &mdash; an experiment that finds no difference has still answered its question.
It says the cheaper model costs you nothing here, which is exactly the sort of claim nobody
should make on the strength of a single demo.

These are stochastic systems, so your numbers will not match mine exactly. Every so often a
run drops an example on the floor &mdash; the specialist writes an artifact with no label in
it, and the supervisor dutifully reports that there is nothing to triage &mdash; and all three
scores go to zero for that one row. That is not the harness misfiring. That is the harness doing the
only job it has.

Passing `metadata={"models": [...]}` fills in the **Models** column in the experiments table.
`max_concurrency=3` keeps things moving without firing every ticket at once.

In [37]:
experiments = []
for agent_model in AGENT_MODELS:
    results = client.evaluate(
        make_target(agent_model),
        data=DATASET_NAME,
        evaluators=[schema_valid, triage_accuracy, summary_quality],
        experiment_prefix="triage-" + agent_model.split(":")[-1],
        description=f"Support-ticket triage on {agent_model}, judged by {JUDGE_MODEL}.",
        metadata={"models": [agent_model]},
        max_concurrency=3,
    )
    experiments.append((agent_model, results))

base_url = experiments[0][1].url.split("/datasets/")[0]
session_ids = ",".join(str(results.experiment_id) for _, results in experiments)
compare_url = f"{base_url}/datasets/{dataset.id}/compare?selectedSessions={session_ids}"

display(Markdown(
    "### Experiments\n\n"
    + "\n".join(
        f"- `{agent_model}` &mdash; [{results.experiment_name}]({results.url})"
        for agent_model, results in experiments
    )
    + f"\n\n**[Compare both experiments side by side]({compare_url})**"
))

View the evaluation results for experiment: 'triage-claude-haiku-4-5-20251001-f565c2a9' at:
https://smith.langchain.com/o/60995f7e-6a4c-4170-99c5-ad8ee020b23f/datasets/b1fb508e-5f28-4220-ac81-c78ff4cf53c1/compare?selectedSessions=a034e618-ccb8-45e3-be26-82427f2319da


View the evaluation results for experiment: 'triage-gpt-5.4-mini-36132b25' at:
https://smith.langchain.com/o/60995f7e-6a4c-4170-99c5-ad8ee020b23f/datasets/b1fb508e-5f28-4220-ac81-c78ff4cf53c1/compare?selectedSessions=fba91b83-4c97-4941-8681-c7cb9842beed




### Experiments

- `anthropic:claude-haiku-4-5-20251001` &mdash; [triage-claude-haiku-4-5-20251001-f565c2a9](https://smith.langchain.com/o/60995f7e-6a4c-4170-99c5-ad8ee020b23f/datasets/b1fb508e-5f28-4220-ac81-c78ff4cf53c1/compare?selectedSessions=a034e618-ccb8-45e3-be26-82427f2319da)
- `openai:gpt-5.4-mini` &mdash; [triage-gpt-5.4-mini-36132b25](https://smith.langchain.com/o/60995f7e-6a4c-4170-99c5-ad8ee020b23f/datasets/b1fb508e-5f28-4220-ac81-c78ff4cf53c1/compare?selectedSessions=fba91b83-4c97-4941-8681-c7cb9842beed)

**[Compare both experiments side by side](https://smith.langchain.com/o/60995f7e-6a4c-4170-99c5-ad8ee020b23f/datasets/b1fb508e-5f28-4220-ac81-c78ff4cf53c1/compare?selectedSessions=a034e618-ccb8-45e3-be26-82427f2319da,fba91b83-4c97-4941-8681-c7cb9842beed)**

# Summary

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:10px; text-align:left;" width="32%">Concept</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">What to remember</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><code style="color:#2b8a3e;">make_target(model)</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Make the target a factory when you want to vary the system under test. One call per
model gives two experiments over one dataset, differing by exactly one thing.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Grade each layer where it lives</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">The specialist writes JSON that code can check; the supervisor writes prose that
only a judge can check. Separate scores tell you <i>which</i> agent broke.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Match the evaluator to the question</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Structure and reference labels are exact, so grade them with code &mdash; cheaper,
faster, and it cannot be talked out of a verdict. Save the model for what code cannot
see.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Pin the judge model</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Vary one thing per experiment. If the grader changes along with the system under
test, the two runs are not measuring the same thing.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>A reference is not an expected response</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><code style="color:#2b8a3e;">outputs.triage</code> fixes the classification and says nothing about
wording, so semantically valid briefs are free to differ.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Inspect before evaluating</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Run the target once and read its real output before writing any extraction or
grading logic. It also proves the subagent, the artifact and the supervisor are wired
together before a full experiment depends on it.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>No difference is still a result</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">The two models scored alike here. That is an answer &mdash; the cheaper one costs
nothing on this task &mdash; not a broken harness. It is also a claim you cannot make from a
single demo.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><code style="color:#2b8a3e;">results.url</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Every <code style="color:#2b8a3e;">evaluate()</code> returns a link into the LangSmith UI, and
joining the experiment ids on <code style="color:#2b8a3e;">?selectedSessions=</code> opens the comparison
view for all of them at once.</td>
</tr>
</tbody>
</table>